# Task 1 NER — JobBERT + CRF (eksperimen pendamping `task1_ner.ipynb`)

**Hipotesis:** sebagian error EDUCATION/EXPERIENCE (fragmen seperti `years`, `to`, batas span salah) muncul karena softmax memilih label tiap token secara independen. CRF memodelkan transisi antar-label sehingga span lebih utuh.

**Langkah**
1. **Diagnosis tanpa training:** pada model final (softmax), hitung transisi BIO tidak valid (`I-X` tanpa `B-X`/`I-X` sebelumnya) dan bandingkan decoding argmax vs **Viterbi dengan larangan transisi tidak valid**.
2. **JobBERT-CRF:** JobBERT + linear + lapisan CRF (transisi dipelajari). Token `IGN` (label parsial) di-*marginalize*: likelihood = jumlah skor semua jalur yang konsisten dengan label yang diketahui (*partial-label CRF*). Data, split, hyperparameter, dan seed sama dengan softmax.

**Aturan keputusan (ditetapkan sebelum run penuh):** tiga metode, masing-masing 2 seed (42, 13): softmax (argmax), softmax + Viterbi terbatas, JobBERT-CRF. Metode yang lebih kompleks hanya dipilih jika rata-rata S val-nya naik **> 0.01** (≈ 2× std antar-seed 0.005) dari metode yang lebih sederhana: softmax → + Viterbi → CRF. Test set hanya dilaporkan.

In [1]:
# Jika dijalankan di Colab / environment baru:
# %pip install -q kagglehub sklearn-crfsuite nltk "transformers>=4.40,<5" torch pandas pyarrow matplotlib seaborn datasets

In [2]:
import os, re, json, random, collections, time, warnings, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
ROOT = Path("..").resolve()                      # notebook berada di notebooks/
DATA, FIG, MODELS = ROOT / "data", ROOT / "figures", ROOT / "models"
for d in (DATA, FIG, MODELS): d.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")

flag = lambda k, default: os.environ.get(k, default) == "1"
RUN_BERT      = flag("RUN_BERT", "1")



SMOKE         = flag("SMOKE", "0")               # subset kecil, hanya untuk uji pipeline

ENTITIES = ["SKILL", "JOB_TITLE", "EDUCATION", "EXPERIENCE"]
MAX_SPAN_WORDS = 10
N_AURA, N_SKILL_ONLY = 500, 1000                 # ukuran tambahan di ablasi tahap 2

def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

C:\Python312\Lib\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.5.4)
  from scipy.stats import gaussian_kde


## 1. Load data
Semua dataset disimpan di `data/` (di-gitignore). Download otomatis jika belum ada.

In [3]:
GREEN = DATA / "green"; GREEN.mkdir(exist_ok=True)
ner_files = list(DATA.glob("kaggle/**/resume-ner-training-dataset/**/train.json"))
if not ner_files:
    os.environ["KAGGLEHUB_CACHE"] = str(DATA / "kaggle")
    import kagglehub; kagglehub.dataset_download("yashpwrr/resume-ner-training-dataset")
    ner_files = list(DATA.glob("kaggle/**/resume-ner-training-dataset/**/train.json"))
for f in ["raw_data/answers.txt", "raw_data/testset.txt", "raw_data/worker_accuracies.csv"]:
    if not (GREEN / Path(f).name).exists():
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/acp19tag/skill-extraction-dataset/main/{f}", GREEN / Path(f).name)

def load_hf(local, name):
    p = DATA / local
    if p.exists(): return pd.read_parquet(p)
    from datasets import load_dataset
    ds_ = load_dataset(name); df_ = pd.concat([ds_[k].to_pandas().assign(split=k) for k in ds_], ignore_index=True)
    df_.to_parquet(p); return df_

cv = json.load(open(ner_files[0], encoding="utf8"))
SURR = re.compile("[" + chr(0xd800) + "-" + chr(0xdfff) + "]")   # lone surrogate -> ganti 1:1 (offset anotasi tetap valid)
for x in cv: x["text"] = SURR.sub(chr(0xfffd), x["text"])
ats = load_hf("resume_ats.parquet", "0xnbk/resume-ats-score-v1-en")
ATS_JOBS = ats.text.str.split(r"\s+SEP\s+", n=1, regex=True).str[1].fillna("").drop_duplicates()
ATS_JOBS = ATS_JOBS[ATS_JOBS.str.len() > 30].tolist()
print("Resume NER:", len(cv), "dokumen | JD unik resume-ats (untuk proxy):", len(ATS_JOBS))

Resume NER: 5960 dokumen | JD unik resume-ats (untuk proxy): 320


## 2. Pilih dokumen CV *rich* + dedup
- *rich* = punya label selain SKILL (sumber DataTurks/Indeed, Resume Corpus, Doccano).
- Dokumen rich yang tidak punya `SKILL` maupun `OTHER` sama sekali dibuang: skill di sana tidak dianotasi sehingga akan jadi `O`.
- Dedup: teks ternormalisasi identik **dan** near-duplicate (300 karakter awal sama). Dari duplikat, simpan yang anotasinya terbanyak.
- Dokumen SKILL-only juga di-dedup (dipakai hanya di ablasi tahap 2).

In [4]:
labsets = [set(a[2] for a in x["annotations"]) for x in cv]
norm = lambda t: re.sub(r"\s+", " ", t.lower()).strip()

def dedup(idx):
    seen_full, seen_pre, out = set(), set(), []
    for i in sorted(idx, key=lambda i: -len(cv[i]["annotations"])):
        t = norm(cv[i]["text"])
        if t in seen_full or t[:300] in seen_pre: continue
        seen_full.add(t); seen_pre.add(t[:300]); out.append(i)
    return sorted(out)

rich = [i for i, s in enumerate(labsets) if s != {"SKILL"}]
no_skill = {i for i in rich if not ({"SKILL", "OTHER"} & labsets[i])}
cv_idx = dedup([i for i in rich if i not in no_skill])
skill_only_idx = dedup([i for i, s in enumerate(labsets) if s == {"SKILL"}])
print(f"rich: {len(rich)} | tanpa SKILL/OTHER (dibuang): {len(no_skill)} | rich setelah dedup: {len(cv_idx)}"
      f" | SKILL-only setelah dedup: {len(skill_only_idx)}")

rich: 988 | tanpa SKILL/OTHER (dibuang): 44 | rich setelah dedup: 824 | SKILL-only setelah dedup: 3274


## 3. Mapping label → skema 4 entity
Representasi internal per contoh: `tokens`, `labels` (salah satu dari ENTITIES / `O` / `IGN`), `sid` (id span, untuk membentuk BIO). `IGN` = tidak dihitung di loss & evaluasi.

Overlap span diselesaikan dengan prioritas: JOB_TITLE > EDUCATION = EXPERIENCE > SKILL > IGN.

In [5]:
CV_MAP = {"SKILL": "SKILL", "LANGUAGE": "SKILL", "DESIGNATION": "JOB_TITLE", "EDUCATION": "EDUCATION",
          "CERTIFICATION": "EDUCATION", "EXPERIENCE": "EXPERIENCE",
          "OTHER": "IGN", "EXPERTISE": "IGN", "ACTION": "IGN", "COLLABORATION": "IGN",
          "PERSON": "O", "EMAIL": "O", "LOCATION": "O", "COMPANY": "O"}
GREEN_MAP = {"Skill": "SKILL", "Occupation": "JOB_TITLE", "Qualification": "EDUCATION", "Experience": "EXPERIENCE", "Domain": "O"}
AURA_MAP = {"TITLE": "JOB_TITLE", "SKILL": "SKILL", "LANGUAGE": "SKILL", "DEGREE": "EDUCATION", "FIELD": "EDUCATION",
            "CERT": "EDUCATION", "DATE": "IGN"}   # DATE ambigu (tanggal kerja / lulus / sertifikat) -> IGN; sisanya O
PRIO = {"IGN": 1, "SKILL": 2, "EXPERIENCE": 3, "EDUCATION": 3, "JOB_TITLE": 4}

TOK = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+|\w+(?:[.+#'-]\w+)*[+#]*|[^\w\s]")

def cap_long_spans(ex):
    """Span entity > MAX_SPAN_WORDS token -> IGN (umumnya blob section, bukan entity)."""
    size = collections.Counter(s for l, s in zip(ex["labels"], ex["sid"]) if l in ENTITIES)
    ex["labels"] = ["IGN" if l in ENTITIES and size[s] > MAX_SPAN_WORDS else l for l, s in zip(ex["labels"], ex["sid"])]
    return ex

def char_example(eid, src, text, anns, mapping):
    """anns: list (start, end, label) offset karakter."""
    toks = [(m.group(), m.start(), m.end()) for m in TOK.finditer(text)]
    st, en = np.array([a for _, a, _ in toks]), np.array([b for _, _, b in toks])
    spans = []
    for s, e, l in anns:
        L = mapping.get(l, "O")
        if L == "O": continue
        span = text[s:e]; s += len(span) - len(span.lstrip()); e -= len(span) - len(span.rstrip())   # trim whitespace
        idx = np.where((st < e) & (en > s))[0] if e > s and len(toks) else []
        if len(idx): spans.append((PRIO[L], L, idx))
    lab, pr, sid = ["O"] * len(toks), [0] * len(toks), [-1] * len(toks)
    for k, (p, L, idx) in enumerate(sorted(spans, key=lambda a: a[0])):
        for t in idx:
            if p >= pr[t]: lab[t], pr[t], sid[t] = L, p, k
    return cap_long_spans({"id": eid, "src": src, "tokens": [t for t, _, _ in toks], "labels": lab, "sid": sid})

def cv_example(i): return char_example(f"cv{i}", "cv", cv[i]["text"], cv[i]["annotations"], CV_MAP)

def bio_example(eid, src, toks, tags, mapping):
    lab, sid, k = [], [], -1
    for j, t in enumerate(tags):
        if t == "O" or "-" not in t: lab.append("O"); sid.append(-1); continue
        p, L = t.split("-", 1)
        if p == "B" or j == 0 or tags[j - 1][2:] != L: k += 1          # I- tanpa B -> span baru
        L2 = mapping.get(L, "O"); lab.append(L2); sid.append(k if L2 != "O" else -1)
    return cap_long_spans({"id": eid, "src": src, "tokens": list(toks), "labels": lab, "sid": sid})

def to_bio(ex):
    lab, sid = ex["labels"], ex["sid"]
    return [l if l in ("O", "IGN") else ("I-" if i and lab[i - 1] == l and sid[i - 1] == sid[i] else "B-") + l
            for i, l in enumerate(lab)]

cv_ex = [cv_example(i) for i in cv_idx]
print(cv_ex[0]["tokens"][:25]); print(to_bio(cv_ex[0])[:25])

['Abhishek', 'Jha', 'Application', 'Development', 'Associate', '-', 'Accenture', 'Bengaluru', ',', 'Karnataka', '-', 'Email', 'me', 'on', 'Indeed', ':', 'indeed.com', '/', 'r', '/', 'Abhishek-Jha', '/', '10e7a8cb732bc43a', '•', 'To']
['O', 'O', 'B-JOB_TITLE', 'I-JOB_TITLE', 'I-JOB_TITLE', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


### Green: agregasi label AMT
`answers.txt`: token + 24 kolom label (satu per pekerja, `?` = tidak menganotasi kalimat itu). Tiap kalimat dianotasi 2 pekerja. Seperti di kode penulis (`find_tag(method='worker_indices')`), label diambil dari pekerja dengan **akurasi kualifikasi tertinggi** di antara yang menganotasi kalimat tersebut. Kalimat yang juga ada di test set dibuang.

In [6]:
acc = pd.read_csv(GREEN / "worker_accuracies.csv").set_index("worker_index").accuracy
RANK = list(acc.sort_values(ascending=False).index)

def read_conll(path):
    sents, cur = [], []
    for line in open(path, encoding="utf8", errors="replace"):
        p = line.split()
        if not p:
            if cur: sents.append(cur); cur = []
        else: cur.append(p)
    if cur: sents.append(cur)
    return sents

def aggregate(rows):
    for w in RANK:
        if all(len(r) > w + 1 and r[w + 1] != "?" for r in rows): return [r[w + 1] for r in rows]
    return None

g_test_raw = [([r[0] for r in s], [r[1] if len(r) > 1 else "O" for r in s]) for s in read_conll(GREEN / "testset.txt")]
test_keys = {" ".join(t).lower() for t, _ in g_test_raw}
g_all, n_dup = [], 0
for s in read_conll(GREEN / "answers.txt"):
    toks, tags = [r[0] for r in s], aggregate(s)
    if tags is None: continue
    if " ".join(toks).lower() in test_keys: n_dup += 1; continue
    g_all.append((toks, tags))
print(f"Green train pool: {len(g_all)} kalimat (dibuang {n_dup} yg ada di test) | Green test: {len(g_test_raw)} kalimat")

Green train pool: 9617 kalimat (dibuang 17 yg ada di test) | Green test: 336 kalimat


### Gaya label SKILL: CV vs Green
Alasan Skill Green perlu perlakuan khusus: separuh span di data AMT berupa frasa/kalimat tugas.

In [7]:
def span_words(exs, L):
    out = []
    for ex in exs:
        b = to_bio(ex)
        for i, t in enumerate(b):
            if t == "B-" + L:
                j = i + 1
                while j < len(b) and b[j] == "I-" + L: j += 1
                out.append(j - i)
    return pd.Series(out)
style = {"CV SKILL": span_words(cv_ex, "SKILL"),
         "Green Skill (AMT)": span_words([bio_example("", "", t, g, GREEN_MAP) for t, g in g_all], "SKILL"),
         "Green Skill (test gold)": span_words([bio_example("", "", t, g, GREEN_MAP) for t, g in g_test_raw], "SKILL")}
display(pd.DataFrame({k: {"n_span": len(v), "median_kata": v.median(), "1-2 kata": f"{(v <= 2).mean():.0%}",
                          ">=4 kata": f"{(v >= 4).mean():.0%}"} for k, v in style.items()}))

,CV SKILL,Green Skill (AMT),Green Skill (test gold)
n_span,10441,11547,656
median_kata,2.0,3.0,2.0
1-2 kata,72%,46%,59%
>=4 kata,14%,42%,28%


## 4. Split, augmentasi, dan varian data
- CV: split level dokumen 80/10/10 (setelah dedup).
- Green: pool AMT → 90/10 train/val; test = gold set penulis.
- Augmentasi `atlas_style`: lowercase + hapus karakter non-alfanumerik (format teks ResumeAtlas di Task 2).
- **Varian Skill Green** (hanya mengubah data *train*):
  - `mask`: semua Skill → IGN
  - `B (semua Skill)`: Skill dipakai apa adanya (tetap kena batas 10 kata)
  - `C (Skill ≤3 kata)` / `C (Skill ≤4 kata)`: Skill pendek → SKILL, yang lebih panjang → IGN

In [8]:
def atlas_style(ex):
    kept = [(re.sub(r"[^a-z0-9]", "", t.lower()), l, s) for t, l, s in zip(ex["tokens"], ex["labels"], ex["sid"])]
    kept = [k for k in kept if k[0]]
    return {**ex, "id": ex["id"] + "_lc", "tokens": [k[0] for k in kept], "labels": [k[1] for k in kept], "sid": [k[2] for k in kept]}

rng = np.random.RandomState(SEED)
perm = rng.permutation(len(cv_ex)); n_te = n_va = int(round(.1 * len(cv_ex)))
cv_test  = [cv_ex[i] for i in perm[:n_te]]
cv_val   = [cv_ex[i] for i in perm[n_te:n_te + n_va]]
cv_train = [cv_ex[i] for i in perm[n_te + n_va:]]
gperm = rng.permutation(len(g_all)); n_gv = int(round(.1 * len(g_all)))
g_val_idx, g_train_idx = gperm[:n_gv], gperm[n_gv:]
g_val  = [bio_example(f"g{i}", "jd", *g_all[i], GREEN_MAP) for i in g_val_idx]          # gold asli (Skill dihitung)
g_test = [bio_example(f"gt{i}", "jd", t, tags, GREEN_MAP) for i, (t, tags) in enumerate(g_test_raw)]
cv_test_lc = [atlas_style(e) for e in cv_test]

if SMOKE:
    cv_train, cv_val, cv_test, cv_test_lc = cv_train[:40], cv_val[:10], cv_test[:10], cv_test_lc[:10]
    g_train_idx, g_val, g_test = g_train_idx[:300], g_val[:60], g_test[:60]
    skill_only_idx, ATS_JOBS = skill_only_idx[:200], ATS_JOBS[:30]
    N_AURA, N_SKILL_ONLY = 30, 30

def green_train(mode):
    out = []
    for i in g_train_idx:
        ex = bio_example(f"g{i}", "jd", *g_all[i], GREEN_MAP)
        size = collections.Counter(s for l, s in zip(ex["labels"], ex["sid"]) if l == "SKILL")
        keep_ = (lambda n: False) if mode == "mask" else (lambda n: True) if mode == "all" else (lambda n: n <= mode)
        ex["labels"] = ["IGN" if l == "SKILL" and not keep_(size[s]) else l for l, s in zip(ex["labels"], ex["sid"])]
        out.append(ex)
    return out

GREEN_VARIANTS = {"mask": "mask", "B (semua Skill)": "all", "C (Skill ≤3 kata)": 3, "C (Skill ≤4 kata)": 4}
CV_BASE = cv_train + [atlas_style(e) for e in cv_train]
TRAINS = {name: CV_BASE + green_train(mode) for name, mode in GREEN_VARIANTS.items()}
VALS  = {"CV val": cv_val, "JD val": g_val}
TESTS = {"CV test": cv_test, "CV test (lowercase)": cv_test_lc, "JD test (gold)": g_test}

def span_counts(exs):
    c = collections.Counter()
    for e in exs:
        b = to_bio(e); c.update(t[2:] for t in b if t.startswith("B-")); c["IGN tokens"] += sum(t == "IGN" for t in b)
    return c
summary = pd.DataFrame({**{f"train: {k}": span_counts(v) for k, v in TRAINS.items()},
                        **{k: span_counts(v) for k, v in {**VALS, **TESTS}.items()}}).fillna(0).astype(int)
display(summary)

,train: mask,train: B (semua Skill),train: C (Skill ≤3 kata),train: C (Skill ≤4 kata),CV val,JD val,CV test,CV test (lowercase),JD test (gold)
JOB_TITLE,8753,8753,8753,8753,456,304,309,309,137
EDUCATION,3517,3517,3517,3517,201,47,175,175,26
EXPERIENCE,5621,5621,5621,5621,396,17,278,278,9
SKILL,16700,27039,22704,23754,1227,1208,864,864,656
IGN tokens,125365,87900,114731,110531,3590,2489,3642,3256,285


### Data tambahan untuk ablasi tahap 2
- **AuraATS** (sintetis): `N_AURA` CV acak dari split train-nya. DEGREE + "in/of" + FIELD digabung jadi satu EDUCATION (gaya CV: *"Bachelor of Science in Statistics"*). Catatan: kosakata sangat sempit (301 skill unik, 110 jabatan unik, 30 template) dan lisensi tidak dicantumkan.
- **SKILL-only**: `N_SKILL_ONLY` dokumen acak dari subset yang hanya berlabel SKILL, dipakai apa adanya.

In [9]:
def merge_degree_field(ex):
    lab, sid, tok = ex["labels"], ex["sid"], ex["tokens"]
    for i in range(1, len(lab) - 1):
        if lab[i] == "O" and tok[i].lower() in ("in", "of") and lab[i - 1] == "EDUCATION" and lab[i + 1] == "EDUCATION":
            old, new = sid[i + 1], sid[i - 1]; lab[i], sid[i] = "EDUCATION", new
            j = i + 1
            while j < len(lab) and lab[j] == "EDUCATION" and sid[j] == old: sid[j] = new; j += 1
    return ex

aura_df = load_hf("aura_synthetic_resume_ner.parquet", "sanaullah7964/AuraATS-Synthetic-Resume-NER-12K-v2")
aura_pool = aura_df[aura_df.split == "train"].sample(N_AURA, random_state=SEED)
AURA = [merge_degree_field(char_example(r.id, "aura", r.text, [(e["start"], e["end"], e["label"]) for e in r.entities], AURA_MAP))
        for r in aura_pool.itertuples()]
SKILL_ONLY = [cv_example(i) for i in np.random.RandomState(SEED).choice(skill_only_idx, min(N_SKILL_ONLY, len(skill_only_idx)), replace=False)]
display(pd.DataFrame({"AuraATS": span_counts(AURA), "SKILL-only": span_counts(SKILL_ONLY)}).fillna(0).astype(int))
for k in range(3):   # cek penggabungan DEGREE + FIELD
    print("contoh EDUCATION/JOB_TITLE Aura:", [(l, t) for t, l in zip(AURA[k]["tokens"], AURA[k]["labels"]) if l in ("EDUCATION", "JOB_TITLE")][:12])

,AuraATS,SKILL-only
JOB_TITLE,1689,0
SKILL,10420,102325
EDUCATION,1020,0
IGN tokens,4587,0


contoh EDUCATION/JOB_TITLE Aura: [('JOB_TITLE', 'Security'), ('JOB_TITLE', 'Analyst'), ('JOB_TITLE', 'Incident'), ('JOB_TITLE', 'Responder'), ('JOB_TITLE', 'Security'), ('JOB_TITLE', 'Engineer'), ('JOB_TITLE', 'GRC'), ('JOB_TITLE', 'Specialist'), ('EDUCATION', 'Bachelor'), ('EDUCATION', 'of'), ('EDUCATION', 'Engineering'), ('EDUCATION', 'in')]
contoh EDUCATION/JOB_TITLE Aura: [('JOB_TITLE', 'Platform'), ('JOB_TITLE', 'Engineer'), ('JOB_TITLE', 'Software'), ('JOB_TITLE', 'Engineer'), ('JOB_TITLE', 'Backend'), ('JOB_TITLE', 'Developer'), ('JOB_TITLE', 'API'), ('JOB_TITLE', 'Engineer'), ('JOB_TITLE', 'Backend'), ('JOB_TITLE', 'Developer'), ('EDUCATION', 'Master'), ('EDUCATION', 'of')]
contoh EDUCATION/JOB_TITLE Aura: [('JOB_TITLE', 'Inventory'), ('JOB_TITLE', 'Manager'), ('JOB_TITLE', 'Warehouse'), ('JOB_TITLE', 'Supervisor'), ('JOB_TITLE', 'Supply'), ('JOB_TITLE', 'Chain'), ('JOB_TITLE', 'Analyst'), ('JOB_TITLE', 'Procurement'), ('JOB_TITLE', 'Specialist'), ('EDUCATION', 'Master'), ('EDU

## 5. Evaluasi
- **exact**: label + batas span sama persis.
- **partial**: label sama dan token overlap (pasangan 1-1). Dipakai untuk JD karena gaya Skill berbeda.
- Prediksi yang menyentuh token `IGN` tidak dihitung sebagai false positive.
- **Proxy recall skill JD (resume-ats)**: dari 28 skill umum yang tertulis di teks JD Task 3, berapa yang ikut terekstrak sebagai SKILL. Kasar (tanpa gold), tapi langsung mengukur kebutuhan Task 3.

In [10]:
def gold_spans(ex):
    out, i, lab, sid = [], 0, ex["labels"], ex["sid"]
    while i < len(lab):
        if lab[i] in ENTITIES:
            j = i + 1
            while j < len(lab) and lab[j] == lab[i] and sid[j] == sid[i]: j += 1
            out.append((lab[i], i, j)); i = j
        else: i += 1
    return out

def bio_spans(tags):
    out, cur = [], None
    for i, t in enumerate(list(tags) + ["O"]):
        if cur and not (t.startswith("I-") and t[2:] == cur[0]): out.append((cur[0], cur[1], i)); cur = None
        if t.startswith("B-") or (t.startswith("I-") and cur is None): cur = (t[2:], i)
    return [s for s in out if s[0] in ENTITIES]

def score(exs, preds, model, dataset):
    st = {m: collections.defaultdict(lambda: [0, 0, 0]) for m in ("exact", "partial")}   # tp, fp, fn
    for ex, tags in zip(exs, preds):
        ign = {i for i, l in enumerate(ex["labels"]) if l == "IGN"}
        G = gold_spans(ex); P = [p for p in bio_spans(tags) if not ign.intersection(range(p[1], p[2]))]
        Gs, Ps = set(G), set(P)
        for g in G: st["exact"][g[0]][0 if g in Ps else 2] += 1
        for p in P:
            if p not in Gs: st["exact"][p[0]][1] += 1
        used = set()
        for g in G:
            k = next((k for k, p in enumerate(P) if k not in used and p[0] == g[0] and p[1] < g[2] and g[1] < p[2]), None)
            if k is None: st["partial"][g[0]][2] += 1
            else: used.add(k); st["partial"][g[0]][0] += 1
        for k, p in enumerate(P):
            if k not in used: st["partial"][p[0]][1] += 1
    rows = []
    for m, d in st.items():
        present = [L for L in ENTITIES if d[L][0] + d[L][2] > 0]          # hanya entity yang ada di gold
        for L in present + (["micro"] if present else []):
            tp, fp, fn = map(sum, zip(*[d[x] for x in present])) if L == "micro" else d[L]
            P_, R_ = tp / max(1, tp + fp), tp / max(1, tp + fn)
            rows.append({"model": model, "dataset": dataset, "match": m, "entity": L, "P": P_, "R": R_,
                         "F1": 2 * P_ * R_ / max(1e-9, P_ + R_), "support": tp + fn})
    return pd.DataFrame(rows)

micro = lambda df, m: df.query(f"match=='{m}' and entity=='micro'").F1.iloc[0] if len(df) else 0.0
SEL_MATCH = {"CV val": "exact", "JD val": "partial"}
def val_parts(predict_fn):
    return {f"{k} F1 ({SEL_MATCH[k]})": micro(score(v, predict_fn([e["tokens"] for e in v]), "", k), SEL_MATCH[k]) for k, v in VALS.items()}

RESULTS, PREDS = [], {}
def evaluate_all(name, predict_fn, show=True):
    for ds, exs in TESTS.items():
        PREDS[(name, ds)] = predict_fn([e["tokens"] for e in exs])
        RESULTS.append(score(exs, PREDS[(name, ds)], name, ds))
    if show:
        r = pd.concat([x for x in RESULTS if x.model.iloc[0] == name])
        display(r.pivot_table(index=["dataset", "entity"], columns="match", values="F1", sort=False).round(3))

PROXY_VOC = ["python", "java", "sql", "excel", "aws", "azure", "javascript", "react", "salesforce", "sap", "tableau", "power bi",
             "linux", "agile", "scrum", "communication", "leadership", "project management", "customer service", "accounting",
             "sales", "marketing", "jira", "git", "docker", "kubernetes", "oracle", "c#"]
def ats_skill_proxy(predict_fn):
    toks = [TOK.findall(t) for t in ATS_JOBS]; hit = miss = 0; n_sk = []
    for t, tags, txt in zip(toks, predict_fn(toks), ATS_JOBS):
        sk = [" ".join(t[s:e]).lower() for L, s, e in bio_spans(tags) if L == "SKILL"]; n_sk.append(len(sk)); joined = " | ".join(sk)
        for v in PROXY_VOC:
            pat = r"(?<![\w.#])" + re.escape(v) + r"(?![\w#])"
            if re.search(pat, txt.lower()):
                if re.search(pat, joined): hit += 1
                else: miss += 1
    n_sk = np.array(n_sk)
    return {"ATS proxy recall": hit / max(1, hit + miss), "ATS JD tanpa SKILL": (n_sk == 0).mean(), "ATS SKILL/JD": n_sk.mean()}

## 6. Fine-tuned JobBERT: perbandingan varian data
- Model: [`jjzha/jobbert-base-cased`](https://huggingface.co/jjzha/jobbert-base-cased) (BERT-base-cased + continued pretraining pada ~3,2 juta kalimat job posting; Zhang et al., NAACL 2022).
- Dokumen CV dipotong menjadi *chunk* ≤ 510 subword tanpa memotong di tengah span (bukan truncation → tidak ada teks yang hilang).
- Label hanya di subword pertama tiap kata; subword lanjutan dan token `IGN` → `-100`.
- Epoch terbaik tiap run dipilih dengan skor validasi **S** (lihat protokol di atas). Hyperparameter sama untuk semua varian.
- **Tahap 1:** 4 varian Skill Green. **Tahap 2:** pemenang tahap 1 + AuraATS, dan pemenang + dokumen SKILL-only. Model final = S tertinggi.

In [11]:
if RUN_BERT:
    import torch
    from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    TAGS = ["O"] + [f"{p}-{L}" for L in ENTITIES for p in "BI"]; T2I = {t: i for i, t in enumerate(TAGS)}
    CFG = dict(lr=3e-5, epochs=1 if SMOKE else 3, batch=4 if DEV == "cuda" else 8, accum=2, max_len=512, warmup=.1, wd=.01)
    JOBBERT = "jjzha/jobbert-base-cased"
    print(DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "", CFG)

    def chunk_bounds(n_sub, sid=None, budget=CFG["max_len"] - 2):
        """Potong daftar kata menjadi chunk <= budget subword; utamakan batas di luar span."""
        out, start, tot = [], 0, 0
        for i, c in enumerate(n_sub):
            if tot + c > budget and i > start:
                cut = i
                if sid is not None:
                    cut = next((j for j in range(i, start, -1) if sid[j] == -1 or sid[j] != sid[j - 1]), i)
                out.append((start, cut)); start, tot = cut, sum(n_sub[cut:i])
            tot += c
        out.append((start, len(n_sub)))
        return out

    def subword_counts(tk, toks):
        if not toks: return []
        wid = tk(toks, is_split_into_words=True, add_special_tokens=False).word_ids()
        return np.bincount([w for w in wid if w is not None], minlength=len(toks)).tolist()

    def encode_examples(tk, exs):
        items = []
        for ex in exs:
            bio = to_bio(ex)
            for s, e in chunk_bounds(subword_counts(tk, ex["tokens"]), ex["sid"]):
                if s == e: continue
                enc = tk(ex["tokens"][s:e], is_split_into_words=True, truncation=True, max_length=CFG["max_len"])
                lab, prev = [], None
                for w in enc.word_ids():
                    lab.append(-100 if w is None or w == prev or bio[s + w] == "IGN" else T2I[bio[s + w]]); prev = w
                if any(l != -100 for l in lab): items.append({"input_ids": enc["input_ids"], "labels": lab})
        return items

    def make_batches(items, bs, shuffle):
        idx = list(range(len(items)))
        if shuffle: random.shuffle(idx)
        mega = [sorted(idx[i:i + bs * 50], key=lambda j: len(items[j]["input_ids"])) for i in range(0, len(idx), bs * 50)]
        bl = [m[i:i + bs] for m in mega for i in range(0, len(m), bs)]
        if shuffle: random.shuffle(bl)
        return bl

    def collate(items, pad_id):
        L = max(len(x["input_ids"]) for x in items)
        ids = torch.full((len(items), L), pad_id); att = torch.zeros((len(items), L), dtype=torch.long)
        lab = torch.full((len(items), L), -100)
        for i, x in enumerate(items):
            n = len(x["input_ids"]); ids[i, :n] = torch.tensor(x["input_ids"]); att[i, :n] = 1
            if "labels" in x: lab[i, :n] = torch.tensor(x["labels"])
        return ids, att, lab

    @torch.no_grad()
    def bert_predict(model, tk, token_lists, bs=16):
        model.eval()
        jobs = []                                                   # (doc, start, input_ids, word_ids)
        for d, toks in enumerate(token_lists):
            for s, e in chunk_bounds(subword_counts(tk, toks)):
                if s == e: continue
                enc = tk(toks[s:e], is_split_into_words=True, truncation=True, max_length=CFG["max_len"])
                jobs.append((d, s, enc["input_ids"], enc.word_ids()))
        out = [["O"] * len(t) for t in token_lists]
        for b in make_batches([{"input_ids": j[2]} for j in jobs], bs, shuffle=False):
            ids, att, _ = collate([{"input_ids": jobs[k][2]} for k in b], tk.pad_token_id)
            with torch.autocast(DEV, enabled=DEV == "cuda"):
                pred = model(input_ids=ids.to(DEV), attention_mask=att.to(DEV)).logits.argmax(-1).cpu().tolist()
            for k, p in zip(b, pred):
                d, s, _, wid = jobs[k]; prev = None
                for pos, w in enumerate(wid):
                    if w is not None and w != prev: out[d][s + w] = TAGS[p[pos]]
                    prev = w
        return out

    def train_bert(ckpt, train_set):
        torch.manual_seed(SEED); random.seed(SEED)
        tk = AutoTokenizer.from_pretrained(ckpt)
        model = AutoModelForTokenClassification.from_pretrained(ckpt, num_labels=len(TAGS), id2label=dict(enumerate(TAGS)), label2id=T2I).to(DEV)
        items = encode_examples(tk, train_set)
        n_steps = CFG["epochs"] * len(make_batches(items, CFG["batch"], False)) // CFG["accum"]
        decay = [p for n, p in model.named_parameters() if not any(x in n for x in ("bias", "LayerNorm"))]
        rest = [p for n, p in model.named_parameters() if any(x in n for x in ("bias", "LayerNorm"))]
        opt = torch.optim.AdamW([{"params": decay, "weight_decay": CFG["wd"]}, {"params": rest, "weight_decay": 0.}], lr=CFG["lr"])
        sch = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * n_steps), n_steps)
        scaler = torch.amp.GradScaler(DEV, enabled=DEV == "cuda")
        hist, best = [], (-1, None)
        print(f"  {len(items)} chunk train, {n_steps} optimizer steps")
        for ep in range(CFG["epochs"]):
            model.train(); t0, tot, nb = time.time(), 0., 0
            for bi, b in enumerate(make_batches(items, CFG["batch"], True)):
                ids, att, lab = collate([items[k] for k in b], tk.pad_token_id)
                with torch.autocast(DEV, enabled=DEV == "cuda"):
                    loss = model(input_ids=ids.to(DEV), attention_mask=att.to(DEV), labels=lab.to(DEV)).loss / CFG["accum"]
                scaler.scale(loss).backward(); tot += loss.item() * CFG["accum"]; nb += 1
                if (bi + 1) % CFG["accum"] == 0:
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update(); opt.zero_grad(); sch.step()
            v = val_parts(lambda L: bert_predict(model, tk, L)); S = np.mean(list(v.values()))
            hist.append({"epoch": ep + 1, "train_loss": tot / nb, **v, "S": S})
            print(f"  epoch {ep + 1}: loss={tot / nb:.4f} " + " ".join(f"{k}={x:.3f}" for k, x in v.items()) + f" S={S:.3f} ({time.time() - t0:.0f}s)")
            if S > best[0]: best = (S, {k: t.detach().cpu().clone() for k, t in model.state_dict().items()})
        model.load_state_dict(best[1])
        return model, tk, pd.DataFrame(hist)

    HIST, COMPARE, BEST = {}, [], {"S": -1}
    def run_variant(name, train_set):
        """Latih, evaluasi (val+test+proxy), simpan hanya model dgn S tertinggi di memori."""
        import gc; gc.collect()
        if DEV == "cuda": torch.cuda.empty_cache()
        print(f"\n### {name}: {len(train_set)} contoh train")
        model, tk, HIST[name] = train_bert(JOBBERT, train_set)
        h = HIST[name].loc[HIST[name].S.idxmax()]
        pf = lambda L: bert_predict(model, tk, L)
        evaluate_all(name, pf, show=False)
        COMPARE.append({"varian": name, "best_epoch": int(h.epoch), **{k: h[k] for k in h.index if "val" in k}, "S (val)": h.S,
                        **ats_skill_proxy(pf)})
        if h.S > BEST["S"]:
            if "model" in BEST: del BEST["model"]
            BEST.update(S=h.S, name=name, model=model, tk=tk, train=train_set)
        else:
            del model

    def compare_table():
        c = pd.DataFrame(COMPARE).set_index("varian")
        r = pd.concat(RESULTS)
        cols = {"test CV micro (exact)": ("CV test", "exact", "micro"), "test CV-lc micro (exact)": ("CV test (lowercase)", "exact", "micro"),
                "test JD micro (partial)": ("JD test (gold)", "partial", "micro"), "test JD SKILL (partial)": ("JD test (gold)", "partial", "SKILL"),
                "test JD JOB_TITLE (exact)": ("JD test (gold)", "exact", "JOB_TITLE")}
        f1 = r.set_index(["model", "dataset", "match", "entity"]).F1
        for col, key in cols.items(): c[col] = [f1.get((v, *key), np.nan) for v in c.index]
        return c

cuda NVIDIA GeForce RTX 3060 Laptop GPU {'lr': 3e-05, 'epochs': 3, 'batch': 4, 'accum': 2, 'max_len': 512, 'warmup': 0.1, 'wd': 0.01}


## 1. Diagnosis: BIO tidak valid & Viterbi terbatas pada model final (softmax)

In [12]:
RUN_CRF = flag("RUN_CRF", "1")
NT, NEG = len(TAGS), -1e9
ALLOW = np.ones((NT, NT), bool)                                     # ALLOW[i, j]: tag i boleh diikuti tag j
for j, t in enumerate(TAGS):
    if t.startswith("I-"): ALLOW[:, j] = [s != "O" and s[2:] == t[2:] for s in TAGS]
START_OK = np.array([not t.startswith("I-") for t in TAGS])
HARD_T, HARD_S, ZERO = np.where(ALLOW, 0., NEG), np.where(START_OK, 0., NEG), np.zeros(NT)

def viterbi(em, trans, start, end):
    """em [T, NT] skor emisi; trans/start/end skor (NEG = dilarang). Kembalikan indeks tag terbaik."""
    score, back = start + em[0], []
    for t in range(1, len(em)):
        cand = score[:, None] + trans                                # [prev, cur]
        back.append(cand.argmax(0)); score = cand.max(0) + em[t]
    best = [int((score + end).argmax())]
    for b in reversed(back): best.append(int(b[best[-1]]))
    return best[::-1]

@torch.no_grad()
def word_scores(emit_fn, tk, token_lists, bs=16):
    """Skor emisi per kata (subword pertama) per chunk: list (doc, start, array[n_kata, NT])."""
    jobs = []
    for d, toks in enumerate(token_lists):
        for s, e in chunk_bounds(subword_counts(tk, toks)):
            if s == e: continue
            enc = tk(toks[s:e], is_split_into_words=True, truncation=True, max_length=CFG["max_len"])
            jobs.append((d, s, enc["input_ids"], enc.word_ids()))
    out = [None] * len(jobs)
    for b in make_batches([{"input_ids": j[2]} for j in jobs], bs, shuffle=False):
        ids, att, _ = collate([{"input_ids": jobs[k][2]} for k in b], tk.pad_token_id)
        with torch.autocast(DEV, enabled=DEV == "cuda"):
            em = emit_fn(ids.to(DEV), att.to(DEV)).float().cpu().numpy()
        for k, m in zip(b, em):
            d, s, _, wid = jobs[k]; pos, prev = [], None
            for p, w in enumerate(wid):
                if w is not None and w != prev: pos.append(p)
                prev = w
            out[k] = (d, s, m[pos])
    return out

def decode(chunks, token_lists, how, trans=None, start=None, end=None):
    tags = [["O"] * len(t) for t in token_lists]
    for d, s, em in chunks:
        idx = em.argmax(1) if how == "argmax" else viterbi(em, trans, start, end)
        for i, k in enumerate(idx): tags[d][s + i] = TAGS[k]
    return tags

def invalid_share(tag_lists):
    """Proporsi tag I-X yang tidak didahului B-X/I-X (di antara semua tag entity)."""
    bad = ent = 0
    for tags in tag_lists:
        prev = "O"
        for t in tags:
            if t != "O":
                ent += 1
                if t.startswith("I-") and (prev == "O" or prev[2:] != t[2:]): bad += 1
            prev = t
    return bad / max(1, ent)

def fragment_stats(tag_lists, token_lists):
    sp = [(L, " ".join(t[s:e]).lower()) for tags, t in zip(tag_lists, token_lists) for L, s, e in bio_spans(tags)]
    exp = [v for L, v in sp if L == "EXPERIENCE"]; edu = [v for L, v in sp if L == "EDUCATION"]
    return {"EXPERIENCE tanpa angka": np.mean([not re.search(r"\d", v) for v in exp]) if exp else np.nan,
            "EDUCATION 1 kata": np.mean([len(v.split()) == 1 for v in edu]) if edu else np.nan,
            "EXP/teks": len(exp) / len(tag_lists), "EDU/teks": len(edu) / len(tag_lists)}

ATS_TOKS = [TOK.findall(t) for t in ATS_JOBS]
SETS = {**{k: [e["tokens"] for e in v] for k, v in {**VALS, **TESTS}.items()}, "ATS JD": ATS_TOKS}

def cached_predictor(emit_fn, tk, dec):
    """predict_fn untuk val_parts/evaluate_all; emisi tiap set dihitung sekali lalu dipakai ulang."""
    cache = {}
    def f(token_lists):
        k = next((k for k, v in SETS.items() if v is token_lists or (len(v) == len(token_lists) and v[:5] == token_lists[:5])), None)
        if k is None: return decode(word_scores(emit_fn, tk, token_lists), token_lists, **dec)
        if k not in cache: cache[k] = word_scores(emit_fn, tk, SETS[k])
        return decode(cache[k], token_lists, **dec)
    return f

def summarize(name, pf, extra=None):
    v = val_parts(pf); evaluate_all(name, pf, show=False)
    return {"model": name, **(extra or {}), **v, "S (val)": np.mean(list(v.values())),
            **{f"BIO tidak valid ({k})": invalid_share(pf(SETS[k])) for k in ["CV test", "JD test (gold)", "ATS JD"]},
            **{f"ATS {k}": x for k, x in fragment_stats(pf(ATS_TOKS), ATS_TOKS).items()}, **ats_skill_proxy(pf)}

def ent_table(names):
    r = pd.concat([x for x in RESULTS if x.model.iloc[0] in names])
    return r.pivot_table(index=["dataset", "entity"], columns=["match", "model"], values="F1", sort=False).round(3)

Seed 42 = model final yang tersimpan (`models/ner_jobbert`). Seed 13 dilatih ulang dengan resep yang sama supaya setiap metode punya 2 seed.

In [13]:
SEEDS = [42, 13]
SM_DEC = {"softmax": dict(how="argmax"), "softmax + Viterbi": dict(how="viterbi", trans=HARD_T, start=HARD_S, end=ZERO)}
ROWS = []
for seed in SEEDS:
    if seed == 42:
        sm = AutoModelForTokenClassification.from_pretrained(MODELS / "ner_jobbert").to(DEV).eval()
        sm_tk = AutoTokenizer.from_pretrained(MODELS / "ner_jobbert")
    else:
        print(f"\n### softmax | seed {seed}")
        SEED = seed; sm, sm_tk, _ = train_bert(JOBBERT, TRAINS["B (semua Skill)"]); SEED = 42
    sm_emit = lambda ids, att: torch.log_softmax(sm(input_ids=ids, attention_mask=att).logits.float(), -1)
    for name, dec in SM_DEC.items():
        ROWS.append(summarize(f"{name} seed {seed}", cached_predictor(sm_emit, sm_tk, dec), {"metode": name, "seed": seed}))
    del sm; import gc; gc.collect(); torch.cuda.empty_cache() if DEV == "cuda" else None
display(pd.DataFrame(ROWS).set_index("model").round(3).T)
display(ent_table([r["model"] for r in ROWS if r["seed"] == 42]))


### softmax | seed 13


Some weights of BertForTokenClassification were not initialized from the model checkpoint at jjzha/jobbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  11532 chunk train, 4324 optimizer steps


  epoch 1: loss=0.4803 CV val F1 (exact)=0.346 JD val F1 (partial)=0.761 S=0.554 (249s)


  epoch 2: loss=0.2712 CV val F1 (exact)=0.405 JD val F1 (partial)=0.767 S=0.586 (271s)


  epoch 3: loss=0.1995 CV val F1 (exact)=0.414 JD val F1 (partial)=0.769 S=0.592 (281s)


model,softmax seed 42,softmax + Viterbi seed 42,softmax seed 13,softmax + Viterbi seed 13
metode,softmax,softmax + Viterbi,softmax,softmax + Viterbi
seed,42,42,13,13
CV val F1 (exact),0.412,0.45,0.414,0.451
JD val F1 (partial),0.768,0.786,0.769,0.785
S (val),0.59,0.618,0.592,0.618
BIO tidak valid (CV test),0.099,0.0,0.108,0.0
BIO tidak valid (JD test (gold)),0.045,0.0,0.051,0.0
BIO tidak valid (ATS JD),0.136,0.0,0.134,0.0
ATS EXPERIENCE tanpa angka,0.322,0.091,0.312,0.078
ATS EDUCATION 1 kata,0.312,0.144,0.356,0.145


match                                    exact         partial                     exact                   partial
model                          softmax seed 42 softmax seed 42 softmax + Viterbi seed 42 softmax + Viterbi seed 42
dataset             entity                                                                                        
CV test             SKILL                0.350           0.534                     0.378                     0.538
                    JOB_TITLE            0.411           0.589                     0.479                     0.614
                    EDUCATION            0.404           0.644                     0.493                     0.728
                    EXPERIENCE           0.477           0.636                     0.539                     0.697
                    micro                0.394           0.577                     0.441                     0.603
CV test (lowercase) SKILL                0.302           0.486                     0.324                     0.495
                    JOB_TITLE            0.393           0.602                     0.434                     0.602
                    EDUCATION            0.353           0.645                     0.435                     0.724
                    EXPERIENCE           0.479           0.666                     0.532                     0.704
                    micro                0.361           0.563                     0.399                     0.581
JD test (gold)      SKILL                0.484           0.770                     0.498                     0.773
                    JOB_TITLE            0.814           0.862                     0.805                     0.861
                    EDUCATION            0.367           0.735                     0.383                     0.766
                    EXPERIENCE           0.381           0.857                     0.421                     0.947
                    micro                0.533           0.785                     0.545                     0.789

## 2. JobBERT-CRF (partial-label)
- Emisi: JobBERT → dropout → linear (sama dengan softmax). Di atasnya matriks transisi `trans[i, j]` + skor awal/akhir, dipelajari.
- Loss = −log P(label yang diketahui) = log Z(semua jalur) − log Z(jalur yang cocok dengan label; posisi `IGN` bebas), dinormalisasi per kata berlabel.
- Decoding: Viterbi dengan transisi yang dipelajari + larangan transisi BIO tidak valid.
- LR BERT & emisi sama dengan softmax (3e-5); parameter CRF 5e-3 (transisi mulai dari nol). 3 epoch, epoch terbaik dipilih dengan S val.

In [14]:
from transformers import AutoModel
CRF_LR = 5e-3

class BertCRF(torch.nn.Module):
    def __init__(self, ckpt):
        super().__init__()
        self.bert = AutoModel.from_pretrained(ckpt, add_pooling_layer=False)
        self.drop, self.emit = torch.nn.Dropout(0.1), torch.nn.Linear(self.bert.config.hidden_size, NT)
        self.trans = torch.nn.Parameter(torch.zeros(NT, NT))
        self.start, self.end = torch.nn.Parameter(torch.zeros(NT)), torch.nn.Parameter(torch.zeros(NT))

    def emissions(self, ids, att):
        return self.emit(self.drop(self.bert(input_ids=ids, attention_mask=att).last_hidden_state))

    def log_z(self, E, m):
        """Forward algorithm. E [B, W, NT], m [B, W] mask kata."""
        a = self.start + E[:, 0]
        for t in range(1, E.shape[1]):
            nxt = torch.logsumexp(a.unsqueeze(2) + self.trans, 1) + E[:, t]
            a = torch.where(m[:, t:t + 1], nxt, a)
        return torch.logsumexp(a + self.end, 1)

    def loss(self, em, lab):
        """em [B, L, NT] emisi subword; lab: -100 = bukan subword pertama, -1 = IGN (semua tag boleh), >=0 = tag gold."""
        em, keep = em.float(), lab != -100
        B, W = len(em), int(keep.sum(1).max())
        E, L = em.new_zeros(B, W, NT), lab.new_full((B, W), -1)
        m = torch.zeros(B, W, dtype=torch.bool, device=em.device)
        for i in range(B):
            n = int(keep[i].sum()); E[i, :n], L[i, :n], m[i, :n] = em[i][keep[i]], lab[i][keep[i]], True
        allowed = (L.unsqueeze(2) == torch.arange(NT, device=em.device)) | (L == -1).unsqueeze(2)
        both = self.log_z(torch.cat([E, E.masked_fill(~allowed, NEG)]), torch.cat([m, m]))
        return (both[:B] - both[B:]).sum() / ((L >= 0) & m).sum().clamp(min=1)

def encode_crf(tk, exs):
    """Seperti encode_examples, tapi IGN = -1 (di-marginalize), bukan -100."""
    items = []
    for ex in exs:
        bio = to_bio(ex)
        for s, e in chunk_bounds(subword_counts(tk, ex["tokens"]), ex["sid"]):
            if s == e: continue
            enc = tk(ex["tokens"][s:e], is_split_into_words=True, truncation=True, max_length=CFG["max_len"])
            lab, prev = [], None
            for w in enc.word_ids():
                lab.append(-100 if w is None or w == prev else -1 if bio[s + w] == "IGN" else T2I[bio[s + w]]); prev = w
            if any(l >= 0 for l in lab): items.append({"input_ids": enc["input_ids"], "labels": lab})
    return items

def crf_decoder(model):
    tr, st, en = (x.detach().float().cpu().numpy() for x in (model.trans, model.start, model.end))
    return dict(how="viterbi", trans=np.where(ALLOW, tr, NEG), start=np.where(START_OK, st, NEG), end=en)

def crf_predict_fn(model, tk):
    def f(token_lists):
        model.eval(); return decode(word_scores(model.emissions, tk, token_lists), token_lists, **crf_decoder(model))
    return f

def train_crf(train_set, seed):
    torch.manual_seed(seed); random.seed(seed)
    tk = AutoTokenizer.from_pretrained(JOBBERT); model = BertCRF(JOBBERT).to(DEV)
    items = encode_crf(tk, train_set)
    n_steps = CFG["epochs"] * len(make_batches(items, CFG["batch"], False)) // CFG["accum"]
    named = dict(model.named_parameters()); crf_names = {"trans", "start", "end"}
    decay = [p for n, p in named.items() if n not in crf_names and not any(x in n for x in ("bias", "LayerNorm"))]
    rest = [p for n, p in named.items() if n not in crf_names and any(x in n for x in ("bias", "LayerNorm"))]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": CFG["wd"]}, {"params": rest, "weight_decay": 0.},
                             {"params": [named[n] for n in crf_names], "lr": CRF_LR, "weight_decay": 0.}], lr=CFG["lr"])
    sch = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * n_steps), n_steps)
    scaler = torch.amp.GradScaler(DEV, enabled=DEV == "cuda")
    hist, best = [], (-1, None)
    print(f"  {len(items)} chunk train, {n_steps} optimizer steps")
    for ep in range(CFG["epochs"]):
        model.train(); t0, tot, nb = time.time(), 0., 0
        for bi, b in enumerate(make_batches(items, CFG["batch"], True)):
            ids, att, lab = (x.to(DEV) for x in collate([items[k] for k in b], tk.pad_token_id))
            with torch.autocast(DEV, enabled=DEV == "cuda"):
                em = model.emissions(ids, att)
            loss = model.loss(em, lab) / CFG["accum"]                 # CRF dihitung di float32
            scaler.scale(loss).backward(); tot += loss.item() * CFG["accum"]; nb += 1
            if (bi + 1) % CFG["accum"] == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sch.step()
        v = val_parts(crf_predict_fn(model, tk)); S = np.mean(list(v.values()))
        hist.append({"epoch": ep + 1, "train_loss": tot / nb, **v, "S": S})
        print(f"  epoch {ep + 1}: loss={tot / nb:.4f} " + " ".join(f"{k}={x:.3f}" for k, x in v.items()) + f" S={S:.3f} ({time.time() - t0:.0f}s)")
        if S > best[0]: best = (S, {k: t.detach().cpu().clone() for k, t in model.state_dict().items()})
    model.load_state_dict(best[1])
    return model, tk, pd.DataFrame(hist)

In [15]:
if SMOKE: CFG["epochs"] = 3                    # smoke: cukup untuk melihat CRF benar-benar belajar
if RUN_CRF:
    for seed in SEEDS:
        import gc; gc.collect(); torch.cuda.empty_cache() if DEV == "cuda" else None
        print(f"\n### JobBERT-CRF | seed {seed}")
        model, tk, hist = train_crf(TRAINS["B (semua Skill)"], seed)
        h = hist.loc[hist.S.idxmax()]
        ROWS.append(summarize(f"CRF seed {seed}", cached_predictor(model.emissions, tk, crf_decoder(model)),
                              {"metode": "CRF", "seed": seed, "best_epoch": int(h.epoch)}))
        if seed == 42 and not SMOKE:
            out = MODELS / "ner_jobbert_crf"; out.mkdir(exist_ok=True)
            torch.save(model.state_dict(), out / "model.pt"); tk.save_pretrained(out)
        del model


### JobBERT-CRF | seed 42


  11532 chunk train, 4324 optimizer steps


  epoch 1: loss=0.3588 CV val F1 (exact)=0.370 JD val F1 (partial)=0.765 S=0.567 (489s)


  epoch 2: loss=0.1311 CV val F1 (exact)=0.440 JD val F1 (partial)=0.802 S=0.621 (497s)


  epoch 3: loss=0.0940 CV val F1 (exact)=0.471 JD val F1 (partial)=0.792 S=0.631 (502s)



### JobBERT-CRF | seed 13


  11532 chunk train, 4324 optimizer steps


  epoch 1: loss=0.3688 CV val F1 (exact)=0.390 JD val F1 (partial)=0.783 S=0.587 (510s)


  epoch 2: loss=0.1316 CV val F1 (exact)=0.455 JD val F1 (partial)=0.802 S=0.629 (515s)


  epoch 3: loss=0.0933 CV val F1 (exact)=0.477 JD val F1 (partial)=0.793 S=0.635 (504s)


## 3. Perbandingan & keputusan

In [16]:
tab = pd.DataFrame(ROWS).set_index("model")
cols = ["S (val)", "CV val F1 (exact)", "JD val F1 (partial)", "BIO tidak valid (CV test)", "BIO tidak valid (JD test (gold))",
        "BIO tidak valid (ATS JD)", "ATS EXPERIENCE tanpa angka", "ATS EDUCATION 1 kata", "ATS EXP/teks", "ATS EDU/teks", "ATS proxy recall", "ATS SKILL/JD"]
display(tab[[c for c in cols if c in tab]].round(3).T)
display(ent_table(list(tab.index)))

agg = tab.groupby("metode")[["S (val)", "CV val F1 (exact)", "JD val F1 (partial)"]].agg(["mean", "std"]).round(3)
display(agg)
S = tab.groupby("metode")["S (val)"].mean()
choice = "softmax"                                                   # metode lebih kompleks hanya dipilih jika naik > 0.01
if S.get("softmax + Viterbi", -1) - S["softmax"] > 0.01: choice = "softmax + Viterbi"
if S.get("CRF", -1) - S[choice] > 0.01: choice = "CRF"
print("S val rata-rata:", S.round(3).to_dict(), "\nKEPUTUSAN:", choice)
if not SMOKE: tab.to_csv(DATA / "t1_crf_check.csv")

model,softmax seed 42,softmax + Viterbi seed 42,softmax seed 13,softmax + Viterbi seed 13,CRF seed 42,CRF seed 13
S (val),0.590,0.618,0.592,0.618,0.631,0.635
CV val F1 (exact),0.412,0.450,0.414,0.451,0.471,0.477
JD val F1 (partial),0.768,0.786,0.769,0.785,0.792,0.793
BIO tidak valid (CV test),0.099,0.000,0.108,0.000,0.000,0.000
BIO tidak valid (JD test (gold)),0.045,0.000,0.051,0.000,0.000,0.000
BIO tidak valid (ATS JD),0.136,0.000,0.134,0.000,0.000,0.000
ATS EXPERIENCE tanpa angka,0.322,0.091,0.312,0.078,0.021,0.036
ATS EDUCATION 1 kata,0.312,0.144,0.356,0.145,0.071,0.060
ATS EXP/teks,1.544,1.138,1.575,1.128,0.875,0.959
ATS EDU/teks,1.553,1.172,1.562,1.144,1.050,0.941


match                                    exact         partial                     exact                   partial           exact         partial                     exact  \
model                          softmax seed 42 softmax seed 42 softmax + Viterbi seed 42 softmax + Viterbi seed 42 softmax seed 13 softmax seed 13 softmax + Viterbi seed 13   
dataset             entity                                                                                                                                                     
CV test             SKILL                0.350           0.534                     0.378                     0.538           0.329           0.512                     0.369   
                    JOB_TITLE            0.411           0.589                     0.479                     0.614           0.422           0.585                     0.470   
                    EDUCATION            0.404           0.644                     0.493                     0.728           0.400           0.644                     0.468   
                    EXPERIENCE           0.477           0.636                     0.539                     0.697           0.459           0.626                     0.537   
                    micro                0.394           0.577                     0.441                     0.603           0.381           0.563                     0.432   
CV test (lowercase) SKILL                0.302           0.486                     0.324                     0.495           0.297           0.483                     0.317   
                    JOB_TITLE            0.393           0.602                     0.434                     0.602           0.390           0.585                     0.434   
                    EDUCATION            0.353           0.645                     0.435                     0.724           0.374           0.655                     0.455   
                    EXPERIENCE           0.479           0.666                     0.532                     0.704           0.442           0.630                     0.504   
                    micro                0.361           0.563                     0.399                     0.581           0.355           0.553                     0.392   
JD test (gold)      SKILL                0.484           0.770                     0.498                     0.773           0.477           0.773                     0.493   
                    JOB_TITLE            0.814           0.862                     0.805                     0.861           0.825           0.879                     0.827   
                    EDUCATION            0.367           0.735                     0.383                     0.766           0.360           0.760                     0.391   
                    EXPERIENCE           0.381           0.857                     0.421                     0.947           0.300           0.900                     0.353   
                    micro                0.533           0.785                     0.545                     0.789           0.529           0.792                     0.545   

match                                            partial       exact     partial       exact     partial  
model                          softmax + Viterbi seed 13 CRF seed 42 CRF seed 42 CRF seed 13 CRF seed 13  
dataset             entity                                                                                
CV test             SKILL                          0.523       0.370       0.515       0.363       0.498  
                    JOB_TITLE                      0.605       0.521       0.635       0.502       0.618  
                    EDUCATION                      0.713       0.544       0.761       0.511       0.748  
                    EXPERIENCE                     0.689       0.638       0.724       0.612       0.732  
                    micro                          0.590       0.470       0.604  

S (val)        CV val F1 (exact)        JD val F1 (partial)       
                     mean    std              mean    std                mean    std
metode                                                                              
CRF                 0.633  0.002             0.474  0.004               0.792  0.001
softmax             0.591  0.001             0.413  0.002               0.769  0.001
softmax + Viterbi   0.618  0.000             0.451  0.001               0.785  0.001

S val rata-rata: {'CRF': 0.633, 'softmax': 0.591, 'softmax + Viterbi': 0.618} 
KEPUTUSAN: CRF


## 4. Kesimpulan
- **Hipotesis terbukti; model final = JobBERT-CRF.** Rata-rata S val (2 seed): softmax 0.591 → softmax + Viterbi terbatas 0.618 (+0.027) → CRF 0.633 (+0.015 dari Viterbi). Kedua kenaikan > 0.01 dan konsisten di dua seed (std ≤ 0.002).
- **Penyebabnya memang urutan label.** Pada softmax, 10% tag entity di CV test (13% di JD ATS) adalah `I-X` tanpa `B-X` sebelumnya. Melarang transisi itu saja (tanpa training) sudah memberi sebagian besar kenaikan. CRF yang dilatih menambah lagi, karena transisi dipelajari dan emisinya dilatih bersama.
- **EDUCATION & EXPERIENCE paling terbantu** (test, seed 42, softmax → CRF):
  - CV test exact: EDUCATION 0.404 → 0.544, EXPERIENCE 0.477 → 0.638. Sekarang di atas baseline CRF fitur manual (0.457 / 0.526). Micro 0.394 → 0.470.
  - JD test partial: micro 0.785 → 0.801; EXPERIENCE 0.857 → 0.941.
  - Fragmen di JD ATS: EXPERIENCE tanpa angka 32% → 2–4%; EDUCATION satu kata 31–36% → 6–7%.
- **Trade-off di SKILL.** SKILL tidak ikut membaik: CV SKILL partial 0.534 → 0.515 / 0.498, dan SKILL per JD ATS turun dari 16 ke 10–11. Proxy recall turun dari 0.645 ke 0.63 / 0.57 (rata-rata 0.60), masih dalam rentang variasi antar-seed softmax (0.50–0.65). CRF lebih konservatif: span lebih utuh, tetapi lebih sedikit.
- **Biaya:** training ~2× lebih lama (~500 s/epoch vs ~260 s), karena forward algorithm dihitung per kata. Inference praktis sama.
- Model seed 42 disimpan di `models/ner_jobbert_crf` (seed 42 ditetapkan sebelum run sebagai model yang disimpan). Inference & export: `scripts/t1_ner_predict.py`.

## 5. Audit error SKILL di CV test (model final CRF, seed 42)
SKILL exact F1 di CV test 0.370 (P 0.385, R 0.356; 864 gold). Error: **345 spurious** (prediksi tanpa gold yang tumpang-tindih), **363 missed**, 183 boundary.
Sampel acak 50 spurious + 50 missed (`random.Random(0)`) dinilai manual oleh satu penilai:

| Kategori | Spurious (50) | Missed (50) |
|---|---|---|
| Gold yang salah / tidak lengkap | **42** — skill asli yang tidak dilabeli di gold (mis. *Data Mining*, *Django Rest Framework*, *Project Management*, *JavaScript*, *SWOT analyses*) | **11** — gold yang bukan skill: judul/header (*Programmer*, *IT Skills*, *Web Technologies*), domain industri (*Healthcare*, *Banking*), potongan kata (*motion*, *underwater*) |
| Ambigu | 4 — kata generik / header kategori (*management*, *Database :*) | 12 — kata kerja & sifat (*mentored*, *Coached*, *Dedicated*), area fungsi (*staffing*) |
| Error model sebenarnya | **4** — bukan skill (*technical skills*, *Computer knowledge*, isi kolom "Industry"), atau daftar skill yang tergabung jadi satu span | **27** — skill asli yang terlewat (*Labour Laws*, *Apex Classes*, *Job Evaluation*, *agile software development*, frasa panjang seperti *Identify and resolve connectivity problems*) |

**Kesimpulan**
- **Sekitar 84% prediksi "spurious" sebenarnya skill asli** (42/50; interval 95% kira-kira 71–92%). Anotasi Resume NER tidak lengkap: banyak skill dilabeli `O`. Presisi SKILL jauh di-*underestimate*.
- **Sekitar separuh "missed" memang kelemahan model** (27/50), terutama skill domain-spesifik (Salesforce Apex, HR) dan frasa deskriptif panjang. Sisanya gold yang bukan skill atau ambigu.
- Error model yang paling khas: **daftar skill tanpa pemisah** (teks yang kehilangan bullet/baris baru) jadi satu span panjang, mis. *Operation System Windows AIX 61 Linux Unix Database Tools DB2*. Ini akan terbantu oleh preprocessing pemisah baris/bullet.
- Perbandingan antar-model di notebook ini **tetap sah**, karena semua model dinilai dengan gold yang sama. Yang terpengaruh adalah angka absolutnya.
- Data latih CV punya celah yang sama (skill asli berlabel `O`), sehingga model belajar melewatkan sebagian skill. Ini menjelaskan recall SKILL yang rendah.